# 5. Классификация IC50 > медианы

Ноутбук решает задачу бинарной классификации для показателя `IC50, mM`.

In [ ]:
# Подключение библиотек

from pathlib import Path
import json
import warnings


import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

random_state = 42
target_columns = ["IC50, mM", "CC50, mM", "SI"]

data_path = Path("/Users/dmitrijgoncarov/mephi_masters/sem_2/classic_ML/coursework/drug_activity_prediction_variant/notebooks/coursework_dataset.csv")
results_dir = Path("/Users/dmitrijgoncarov/mephi_masters/sem_2/classic_ML/coursework/drug_activity_prediction_variant/notebooks/results")
figures_dir = results_dir / "figures"
tables_dir = results_dir / "tables"
figures_dir.mkdir(parents=True, exist_ok=True)
tables_dir.mkdir(parents=True, exist_ok=True)


from sklearn.ensemble import ExtraTreesClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    accuracy_score,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.svm import SVC


In [ ]:
# Загрузка данных

dataset = pd.read_csv(data_path)
dataset = dataset.drop(columns=[column for column in dataset.columns if str(column).startswith("Unnamed")], errors="ignore")
dataset = dataset.apply(pd.to_numeric, errors="coerce")
dataset = dataset.drop_duplicates()
dataset = dataset.dropna(subset=target_columns).reset_index(drop=True)

print("Размер датасета после базовой очистки:", dataset.shape)
print(dataset.head())


In [ ]:
# Подготовка признаков и бинарной целевой переменной

target_column = 'IC50, mM'
threshold_setting = 'median'
task_id = 'classification_ic50_median'

if threshold_setting == "median":
    threshold = float(dataset[target_column].median())
else:
    threshold = float(threshold_setting)

y = (dataset[target_column] > threshold).astype(int)
feature_columns = [column for column in dataset.columns if column not in target_columns]
x = dataset[feature_columns]

print("Целевая переменная:", target_column)
print("Порог классификации:", threshold)
print("Количество признаков:", x.shape[1])
print("Доля положительного класса:", round(float(y.mean()), 3))
print(y.value_counts().rename("count").to_frame())


## Аналитический вывод

Положительный класс соответствует соединениям, у которых IC50 выше медианы. С практической точки зрения это означает разделение веществ на группы с более низкой и более высокой концентрацией подавления активности.

Классы почти сбалансированы, поэтому accuracy можно интерпретировать достаточно прямо. Тем не менее ROC-AUC остается основной метрикой, так как показывает качество ранжирования соединений независимо от конкретного порога классификации.

In [ ]:
# Разбиение данных и описание конвейера обучения

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=random_state,
    stratify=y,
)


def make_pipeline(model, scale=False):
    # Конвейер фиксирует одинаковую предобработку для всех моделей.
    # Заполнение пропусков выполняется внутри кросс-валидации.
    steps = [
        ("imputer", SimpleImputer(strategy="median")),
        ("variance", VarianceThreshold()),
    ]
    if scale:
        steps.append(("scaler", RobustScaler()))
    steps.append(("model", model))
    return Pipeline(steps)


models = {
    "LogisticRegression": (
        make_pipeline(
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                solver="liblinear",
                random_state=random_state,
            ),
            scale=True,
        ),
        {"model__C": [0.1, 1.0, 5.0]},
    ),
    "SVC": (
        make_pipeline(SVC(class_weight="balanced", random_state=random_state), scale=True),
        {"model__C": [1.0, 5.0], "model__gamma": ["scale"]},
    ),
    "RandomForest": (
        make_pipeline(RandomForestClassifier(random_state=random_state, n_jobs=-1, class_weight="balanced"), scale=False),
        {"model__n_estimators": [120], "model__max_depth": [None, 12]},
    ),
    "ExtraTrees": (
        make_pipeline(ExtraTreesClassifier(random_state=random_state, n_jobs=-1, class_weight="balanced"), scale=False),
        {"model__n_estimators": [140], "model__max_depth": [None, 12]},
    ),
    "GradientBoosting": (
        make_pipeline(GradientBoostingClassifier(random_state=random_state), scale=False),
        {"model__n_estimators": [100, 160], "model__learning_rate": [0.05, 0.1]},
    ),
}


def positive_scores(estimator, x_values):
    # ROC-AUC требует не жесткий класс, а числовую оценку принадлежности к положительному классу.
    if hasattr(estimator, "predict_proba"):
        return estimator.predict_proba(x_values)[:, 1]
    return estimator.decision_function(x_values)


In [ ]:
# Обучение моделей и подбор гиперпараметров

cross_validation = StratifiedKFold(n_splits=3, shuffle=True, random_state=random_state)
metric_rows = []
best_estimator = None
best_roc_auc = -np.inf

for model_name, (model, param_grid) in models.items():
    print("Обучение модели:", model_name)

    search = GridSearchCV(
        model,
        param_grid,
        scoring="roc_auc",
        cv=cross_validation,
        n_jobs=-1,
    )
    search.fit(x_train, y_train)

    prediction = search.best_estimator_.predict(x_test)
    score = positive_scores(search.best_estimator_, x_test)
    roc_auc = roc_auc_score(y_test, score)

    metric_rows.append(
        {
            "model": model_name,
            "threshold": threshold,
            "cv_roc_auc": search.best_score_,
            "test_roc_auc": roc_auc,
            "test_accuracy": accuracy_score(y_test, prediction),
            "test_f1": f1_score(y_test, prediction),
            "best_params": json.dumps(search.best_params_, ensure_ascii=False),
        }
    )

    if roc_auc > best_roc_auc:
        best_roc_auc = roc_auc
        best_estimator = search.best_estimator_

metrics = pd.DataFrame(metric_rows).sort_values("test_roc_auc", ascending=False).reset_index(drop=True)
metrics_path = tables_dir / f"{task_id}_metrics.csv"
metrics.to_csv(metrics_path, index=False)

print(metrics)
print("Метрики сохранены:", metrics_path)


## Вывод по результатам моделирования

Лучший результат показала модель GradientBoosting: ROC-AUC = 0.817, accuracy = 0.732, F1 = 0.737. Это означает, что модель достаточно хорошо отделяет соединения с IC50 выше медианы от соединений с IC50 ниже или равным медиане.

Качество классификации заметно лучше, чем качество прямой регрессии IC50. Это логично: разделить соединения на две группы проще, чем точно восстановить числовое значение IC50, особенно при наличии выбросов.

In [ ]:
# Сравнение моделей по ROC-AUC

plot_data = metrics.sort_values("test_roc_auc", ascending=False)

plt.figure(figsize=(8, 4))
plt.barh(plot_data["model"], plot_data["test_roc_auc"])
plt.gca().invert_yaxis()
plt.xlabel("Test ROC-AUC")
plt.title(f"Сравнение моделей: {task_id}")
plt.tight_layout()
comparison_path = figures_dir / f"{task_id}_model_comparison.png"
plt.savefig(comparison_path, dpi=160)
plt.show()

print("График сохранен:", comparison_path)


In [ ]:
# Диагностика лучшей модели

figure, axes = plt.subplots(1, 2, figsize=(11, 4))

ConfusionMatrixDisplay.from_estimator(best_estimator, x_test, y_test, ax=axes[0], colorbar=False)
axes[0].set_title("Матрица ошибок")

RocCurveDisplay.from_estimator(best_estimator, x_test, y_test, ax=axes[1])
axes[1].set_title("ROC-кривая")

plt.tight_layout()
diagnostic_path = figures_dir / f"{task_id}_diagnostics.png"
plt.savefig(diagnostic_path, dpi=160)
plt.show()

print("График сохранен:", diagnostic_path)
print("Лучшая модель:", metrics.loc[0, "model"])


## Итоговый аналитический вывод

Классификация IC50 относительно медианы является практически применимой задачей. Модель может использоваться для предварительного ранжирования соединений по уровню активности, но для точного выбора кандидатов желательно дополнительно анализировать вероятности классов и настраивать порог решения.